# SDK End-to-End Workflow Test

This notebook tests the complete Graph OLAP SDK workflow from a Jupyter analyst's perspective:

1. Create a mapping via SDK
2. Create a snapshot via SDK (triggers async export)
3. Wait for snapshot to be ready
4. Create an instance via SDK
5. Wait for instance to be running
6. Connect and execute Cypher queries

**Prerequisites:**
- Running in k3d cluster with all services deployed
- SDK installed (`pip install graph-olap`)
- Environment variables set: `CONTROL_PLANE_URL`, `WRAPPER_URL`

## Setup

In [ ]:
import os
import time

# Configuration from environment
CONTROL_PLANE_URL = os.environ.get("CONTROL_PLANE_URL", "http://control-plane:8080")
WRAPPER_URL = os.environ.get("WRAPPER_URL", "http://ryugraph-wrapper:8000")
PUBSUB_EMULATOR_HOST = os.environ.get("PUBSUB_EMULATOR_HOST", "pubsub-emulator:8085")
GCP_PROJECT = os.environ.get("GCP_PROJECT", "test-project")

# Timeouts
SNAPSHOT_TIMEOUT = int(os.environ.get("SNAPSHOT_TIMEOUT", "300"))
INSTANCE_TIMEOUT = int(os.environ.get("INSTANCE_TIMEOUT", "180"))
POLL_INTERVAL = int(os.environ.get("POLL_INTERVAL", "5"))

print(f"Control Plane URL: {CONTROL_PLANE_URL}")
print(f"Wrapper URL: {WRAPPER_URL}")

In [ ]:
# Import SDK
from graph_olap import GraphOLAPClient
from graph_olap.instance.connection import InstanceConnection
from graph_olap.models.mapping import (
    EdgeDefinition,
    NodeDefinition,
    PropertyDefinition,
)

print("SDK imported successfully")

## Create SDK Client

In [ ]:
client = GraphOLAPClient(
    api_url=CONTROL_PLANE_URL,
    username="e2e-test-user",
)
print(f"SDK client created for {CONTROL_PLANE_URL}")

## Phase 1: Create Mapping

In [ ]:
# Define Person node (matches starburst-mock test data)
person_node = NodeDefinition(
    label="Person",
    sql="SELECT id, name, age FROM analytics.public.persons",
    primary_key={"name": "id", "type": "INT64"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ],
)

# Define KNOWS edge
knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT from_id, to_id, since FROM analytics.public.knows",
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ],
)

print("Definitions created:")
print(f"  - Person node with {len(person_node.properties)} properties")
print(f"  - KNOWS edge with {len(knows_edge.properties)} properties")

In [ ]:
# Create mapping via SDK
mapping = client.mappings.create(
    name="SDK E2E Test Mapping",
    description="Mapping for SDK E2E workflow testing",
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)

print(f"Mapping created: {mapping.name}")
print(f"  ID: {mapping.id}")
print(f"  Version: {mapping.current_version}")
mapping

In [ ]:
# Verify mapping
retrieved = client.mappings.get(mapping.id)
assert retrieved.id == mapping.id
assert len(retrieved.node_definitions) == 1
assert len(retrieved.edge_definitions) == 1
print("Mapping verified")

## Phase 2: Create Snapshot

Creating a snapshot triggers an async export process:
1. SDK creates snapshot record (status='pending')
2. Control Plane publishes to Pub/Sub
3. Export Submitter queries Starburst, schedules poll tasks
4. Export Poller monitors queries, writes Parquet to GCS
5. Export Poller updates snapshot status to 'ready'

In [ ]:
# Create snapshot via SDK
snapshot = client.snapshots.create(
    mapping_id=mapping.id,
    name="SDK E2E Test Snapshot",
    description="Snapshot for SDK E2E workflow testing",
)

print(f"Snapshot created: {snapshot.name}")
print(f"  ID: {snapshot.id}")
print(f"  Status: {snapshot.status}")
print(f"  GCS path: {snapshot.gcs_path}")
snapshot

In [ ]:
# Trigger export via Pub/Sub (simulates what Control Plane does in production)
# In a real deployment, Control Plane publishes automatically
import json

from google.cloud import pubsub_v1

os.environ["PUBSUB_EMULATOR_HOST"] = PUBSUB_EMULATOR_HOST
publisher = pubsub_v1.PublisherClient()
topic_path = f"projects/{GCP_PROJECT}/topics/snapshot-requests"

message_data = json.dumps({
    "snapshot_id": snapshot.id,
    "mapping_id": mapping.id,
    "gcs_path": snapshot.gcs_path,
}).encode("utf-8")

future = publisher.publish(topic_path, message_data)
message_id = future.result()
print(f"Published export request: {message_id}")

In [ ]:
# Wait for snapshot to become ready via SDK
print(f"Waiting for snapshot to be ready (timeout={SNAPSHOT_TIMEOUT}s)...")
snapshot = client.snapshots.wait_until_ready(
    snapshot.id,
    timeout=SNAPSHOT_TIMEOUT,
    poll_interval=POLL_INTERVAL,
)

print(f"Snapshot ready: {snapshot.name}")
print(f"  Status: {snapshot.status}")
print(f"  Node counts: {snapshot.node_counts}")
print(f"  Edge counts: {snapshot.edge_counts}")
snapshot

In [ ]:
# Verify snapshot data
assert snapshot.status == "ready"
assert snapshot.node_counts is not None
assert "Person" in snapshot.node_counts
assert snapshot.node_counts["Person"] == 5
assert snapshot.edge_counts is not None
assert "KNOWS" in snapshot.edge_counts
assert snapshot.edge_counts["KNOWS"] == 5
print("Snapshot verified: 5 Person nodes, 5 KNOWS edges")

## Phase 3: Create Instance

Creating an instance triggers wrapper startup:
1. SDK creates instance record (status='starting')
2. Wrapper loads data from GCS
3. Wrapper creates graph schema
4. Wrapper reports 'running' status

In [ ]:
# Create instance via SDK (blocking call - waits until running)
print(f"Creating instance (timeout={INSTANCE_TIMEOUT}s)...")

def on_progress(phase, completed, total):
    print(f"  Progress: {phase} ({completed}/{total})")

instance = client.instances.create_and_wait(
    snapshot_id=snapshot.id,
    name="SDK E2E Test Instance",
    description="Instance for SDK E2E workflow testing",
    timeout=INSTANCE_TIMEOUT,
    poll_interval=POLL_INTERVAL,
    on_progress=on_progress,
)

print(f"Instance running: {instance.name}")
print(f"  ID: {instance.id}")
print(f"  Status: {instance.status}")
print(f"  URL: {instance.instance_url}")
instance

In [ ]:
# Verify instance
assert instance.status == "running"
print("Instance verified: status=running")

## Phase 4: Connect and Query

In [ ]:
# Connect to instance via SDK
# Note: In k3d, use WRAPPER_URL as the instance_url may be internal
conn = InstanceConnection(
    instance_url=WRAPPER_URL,
    instance_id=instance.id,
)

# Wait for connection to be ready
for attempt in range(30):
    try:
        status = conn.status()
        if status.get("ready", False) or "instance_id" in status:
            print("Connection ready")
            break
    except Exception:
        if attempt >= 29:
            raise
        time.sleep(2)

In [ ]:
# Simple node count query
result = conn.query("MATCH (n) RETURN count(n) AS count")
node_count = result.rows[0][0]

print(f"Total nodes: {node_count}")
assert node_count == 5, f"Expected 5 nodes, got {node_count}"

In [ ]:
# Query Person nodes
result = conn.query(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name"
)

print(f"Person nodes: {result.row_count}")
for row in result.rows:
    print(f"  {row[0]}: age {row[1]}")

assert result.row_count == 5
names = [row[0] for row in result.rows]
assert names == ["Alice", "Bob", "Charlie", "Diana", "Eve"]

In [ ]:
# Query KNOWS relationships
result = conn.query("""
    MATCH (a:Person)-[r:KNOWS]->(b:Person)
    RETURN a.name AS from_person, b.name AS to_person, r.since AS since
    ORDER BY a.name, b.name
""")

print(f"KNOWS relationships: {result.row_count}")
for row in result.rows:
    print(f"  {row[0]} -> {row[1]} (since {row[2]})")

assert result.row_count == 5

In [ ]:
# DataFrame query
df = conn.query_df(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.age DESC"
)

print("Person DataFrame:")
print(df)

assert len(df) == 5
ages = df["age"].to_list()
assert ages == [35, 32, 30, 28, 25], f"Expected [35, 32, 30, 28, 25], got {ages}"

In [ ]:
# Scalar query
avg_age = conn.query_scalar("MATCH (p:Person) RETURN avg(p.age)")
print(f"Average age: {avg_age}")
assert avg_age == 30, f"Expected 30, got {avg_age}"

In [ ]:
# Aggregation query
result = conn.query("""
    MATCH (p:Person)
    RETURN
        avg(p.age) AS avg_age,
        min(p.age) AS min_age,
        max(p.age) AS max_age,
        count(p) AS total
""")

row = result.rows[0]
print("Aggregations:")
print(f"  avg_age: {row[0]}")
print(f"  min_age: {row[1]}")
print(f"  max_age: {row[2]}")
print(f"  total: {row[3]}")

assert row[0] == 30  # avg
assert row[1] == 25  # min (Bob)
assert row[2] == 35  # max (Charlie)
assert row[3] == 5   # total

In [ ]:
# Path query
result = conn.query("""
    MATCH (a:Person)-[:KNOWS*1..2]->(b:Person)
    WHERE a <> b
    RETURN DISTINCT a.name AS from_person, b.name AS to_person
    ORDER BY from_person, to_person
""")

print(f"Path query: {result.row_count} connections within 2 hops")

In [ ]:
# Schema inspection
schema = conn.get_schema()

print("Graph Schema:")
print(f"  Node labels: {list(schema.node_labels.keys())}")
print(f"  Relationship types: {list(schema.relationship_types.keys())}")

assert "Person" in schema.node_labels
assert "KNOWS" in schema.relationship_types

## Cleanup

In [ ]:
# Close connection
conn.close()
client.close()
print("Connections closed")

## Summary

In [ ]:
print("=" * 60)
print("SDK E2E WORKFLOW COMPLETE")
print("=" * 60)
print(f"Mapping ID: {mapping.id}")
print(f"Snapshot ID: {snapshot.id}")
print(f"Instance ID: {instance.id}")
print("Nodes loaded: 5")
print("Edges loaded: 5")
print("Average age: 30")
print("=" * 60)
print("All assertions passed!")